# Reading From Bronze

In the silver layer, we will only clean the data for each table. (Merging / data modeling will be done in the Gold Layer)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Data Transformation

Data catelog description from BTS:
| FIELD NAMES | DESCRIPTION |
| ----------- | ----------- |
| Code | airline code |
| Description | airline carrier name |



In [0]:
df_airlines = spark.table("bronze_travel.airlines")

## Exploring raw data

In [0]:
df_airlines.display()

In [0]:
# Check data types

df_airlines.dtypes

**Clean Up To Do List:**

- Trim white spaces on every column
- Remove duplicates
- key columns cannot be null: "Code", "Description"
- Check for NULLs
- Rename columns

## Trim white spaces from all columns

In [0]:
for column_name, data_type in df_airlines.dtypes:
  if data_type == "string":
    df_airlines = df_airlines.withColumn(column_name, F.trim(F.col(column_name)))

## Drop Duplicates

In [0]:
df_airlines.columns

In [0]:
# Find groups with more than one row
dup_groups = (
    df_airlines.groupBy(*df_airlines.columns)
    .count()
    .filter(F.col("count") > 1)
)

# Number of *extra* rows that dropDuplicates will remove
# (count - 1 per group, summed across all groups)
dup_row_count = (
    dup_groups
    .withColumn("extra_rows", F.col("count") - 1)
    .agg(F.sum("extra_rows"))
    .first()[0]
) or 0  # guard against None if there are no duplicates at all

print(f"Found {dup_row_count} duplicate rows across {dup_groups.count()} groups")

before_count = df_airlines.count()

# Drop duplicates, keeping one row per group
df_airlines = df_airlines.dropDuplicates(df_airlines.columns)

after_count = df_airlines.count()
actual_dropped = before_count - after_count

print(f"Before: {before_count}, After: {after_count}, Dropped: {actual_dropped}")

if actual_dropped != dup_row_count:
    raise ValueError(
        f"Expected to drop {dup_row_count} duplicate rows but dropped {actual_dropped}. "
    )

## Check for NULL Values

In [0]:
# Checking for NULLs

for column in df_airlines.columns:
    null_count = df_airlines.select(F.sum(F.col(column).isNull().cast("int")).alias("count")).first()["count"]

    if null_count > 0:
        raise ValueError(f"Column {column} has {null_count} NULL values. Please investigate.")

## Renaming Columns

In [0]:
# Rename columns

rename_columns = {
    "Code": "code",
    "Description": "name"
}

In [0]:
# Updating column names
for old_column, new_column in rename_columns.items():
    df_airlines = df_airlines.withColumnRenamed(old_column, new_column)

# Write Into Silver Table

In [0]:
df_airlines.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_travel.airlines")

In [0]:
%sql
SELECT *
FROM silver_travel.airlines
LIMIT 100